In [28]:
from protevo.models import load_model
from esm.data import Alphabet
vocab = Alphabet.from_architecture("ESM-1b")

from protevo.simulation import simulate_evolution_with_rejection_sampling_batched
from ete3 import Tree
import torch

from functools import partial
from protevo.simulation._simulate_on_tree import rejection_filter

## Load the inputs

Read in the tree file and root sequence

In [ ]:
model, vocab = load_model('model_checkpoints/peint.ckpt', use_cached_model=True, device='cuda', use_flash=True)

In [ ]:
tree = Tree('protevo/tests/simulation_test_dir/tree_dir/1ekj_1_C.newick', format=1)

for n in tree.traverse():
    n.add_features(treename= '1ekj_1_C')

with open('protevo/tests/simulation_test_dir/root_sequences_dir/1ekj_1_C.txt', 'r') as fin:
    label = next(fin)
    seq = next(fin).rstrip('\n')


simulation_args = {
    'model': model,
    'root_sequences': {'1ekj_1_C': seq},
    'trees': [tree],
    'device': torch.device('cuda'),
    'vocab': vocab,
    'max_decode_steps': 2*len(seq), #this would be twice the length of the longest seq if multiple trees/roots are provided
    'max_batch_size': 32,
    'rejection_sampling_length_ratio': {'1ekj_1_C': partial(rejection_filter, length=len(seq), ratio=0.1)},
}

In [30]:
output = simulate_evolution_with_rejection_sampling_batched(
    **simulation_args
)

In [32]:
output['1ekj_1_C']['seq115']

'TDFSFQNEAWERLQNGFVHFRNNVYNKNPSLFARLSPGQSPKFLIFSCSDSRVCPSKILDLDPGESFVVRNVANLVPPNNEGQYAGSGAAMDYAVWSLKVKNIVVIGHSSCGGIKGLLSFPFDGNNTTEFIEEWIKIGLPAREKSVEMFGLGGNDIFFKRCEEEAIRVSLTNLMTCPFVQEAFDQGLVVKGGYYDFVTGRFMLLDLESGEFEP'